In [3]:
import numpy as np
import scipy.sparse as sp

from math import sqrt

import igl
import pyvista as pv
pv.set_jupyter_backend('trame')

In [4]:
def to_pyvista_mesh(V, F):
    return pv.UnstructuredGrid({pv.CellType.TRIANGLE: F}, V)

In [5]:
v, f = igl.read_triangle_mesh("reconstructed_mesh.obj")

In [6]:
mesh = to_pyvista_mesh(v, f)
mesh.plot(show_edges=True)

Widget(value='<iframe src="http://localhost:63825/index.html?ui=P_0x12081c940_0&reconnect=auto" class="pyvista…

In [7]:
# each idx corresponds to a face, each element is a component, highest component count = biggest component\
# Filter out noise components
(num_components, component_ids) = igl.facet_components(f)
unique, counts = np.unique(component_ids, return_counts=True)
count_per_component = dict(zip(unique, counts))

comp_vals = list(count_per_component.values())
largest_comp_idx = comp_vals.index(max(comp_vals))
filtered_f_idx = [i for i, comp in enumerate(component_ids) if comp == largest_comp_idx]

filtered_f = f[filtered_f_idx, :]

to_pyvista_mesh(v, filtered_f).plot(show_edges=True)

Widget(value='<iframe src="http://localhost:63825/index.html?ui=P_0x16c2f27f0_1&reconnect=auto" class="pyvista…

In [20]:
vf, ni = igl.vertex_triangle_adjacency(f, len(v))
areas = igl.doublearea(v, f)
angles = igl.internal_angles(v, f)

print(angles[1, 0] + angles[1, 1] + angles[1, 2])

A_flat = []
for i in range(len(v)):
    adj_faces = []
    # Find all incident faces for current vertex
    for j in range(0, ni[i+1] - ni[i]):
        adj_faces.append(vf[ni[i]+j])
    A_i = 1/3 * sum(areas[adj_faces])
    A_flat.append(A_i)

assert(len(A_flat) == len(v))
A = np.diag(A_flat)

3.141592653589793
